# Basketball-Reference Players Fetcher

we will fetch 4 tables:
1. player (name, position, height, weight, ...)
2. player details (shoots, nationality, team, ...)
3. NBA Michael Jordan award winners (season, league, player, games, ...)
4. Ranking of players in each season based on points (rank, player,  points, ...)

In [27]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup as bs
from selenium.webdriver import ActionChains
from selenium.webdriver.chrome.options import Options

import json

import pandas as pd
import numpy as np
import re

import threading
import warnings

warnings.filterwarnings('ignore')

## Player

This table will be extracted from [basketball-reference](https://www.basketball-reference.com/players/) players page.

In [ ]:
chrome_options = Options()
chrome_options.add_argument("--headless")
driver = webdriver.Chrome(options=chrome_options)

players_csv = ''
letters = 'abcdefghijklmnopqrstuvwyz'
index = 0

while index < 25:

    try:
        driver.get(f'https://www.basketball-reference.com/players/{letters[index]}/')
        WebDriverWait(driver, 5).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, '[data-row]')))

        share_button = driver.find_element(By.XPATH, '//*[@id="players_sh"]/div/ul/li[1]')
        ActionChains(driver).move_to_element(share_button).perform()

        csv_button = driver.find_elements(By.CLASS_NAME, 'tooltip')[2]
        csv_button.click()

        players = driver.find_element(By.ID, 'csv_players')
        players = players.get_attribute('innerHTML')
    
        players_csv += players[players.find('Player'):] + '\n'

        index += 1

    except:
        continue

In [ ]:
file = open('players.csv', 'w+', encoding='utf:8')
file.write(players_csv)

In [ ]:
players = pd.read_csv('players.csv')

In [ ]:
players.drop(players[players['Player'] == 'Player'].index, inplace=True)
players.to_csv('players.csv', index=False)

## Player Detail

    Get data from website

This data will be downloaded from these [pages](https://www.basketball-reference.com/players/a/) for different alphabets.

In [6]:
def get_player(driver, player_id):

    driver.get(f'https://www.basketball-reference.com/players/{player_id[0]}/{player_id}.html')
    driver.implicitly_wait(3)

    iter = 0
    fetched = False

    while iter < 5:

        try:
            element = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'meta_more_button')))
            element.click()
        
        except:
            pass
        
        try:
            player_info = WebDriverWait(driver, 60).until(EC.presence_of_element_located((By.ID, 'info'))).text

            if 'More bio, uniform, draft info' in player_info:
                continue
            
            to_scrape.append((player_id, player_info))
            fetched = True
            break


        except:
            driver.refresh()
            iter += 1

    if not fetched:
        didnt_fetch.append(player_id)

we will get data parallel and in seperated chunks

In [7]:
def get_players_batch(pl_ids):
    chrome_options = Options()
    chrome_options.page_load_strategy = 'none'
    chrome_options.add_argument("--headless")
    chrome_options.add_argument('--user-agent="Mozilla/5.0 (Windows Phone 10.0; Android 4.2.1; Microsoft; Lumia 640 XL LTE) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/42.0.2311.135 Mobile Safari/537.36 Edge/12.10166"')
    driver = webdriver.Chrome(options=chrome_options)

    for player_id in pl_ids:
        get_player(driver, player_id)

In [ ]:
to_scrape = []
didnt_fetch = []

def get_player_details(cuts):

    global to_scrape
    global didnt_fetch

    threads = []
    for cut in range(len(cuts)-1):
        t = threading.Thread(target=get_players_batch, args=(players[cuts[cut]:cuts[cut+1]]['Player-additional']))
        threads.append(t)

    for t in threads:
        t.start()

    for t in threads:
        t.join()
        
    return to_scrape, didnt_fetch

    Extract fields

we will extract different fields to create a table.

In [ ]:
def extract_player_details(to_scrape):
    
    players_details = pd.DataFrame({
        'player_id': [],
        'position': [],
        'shoots': [],
        'height': [],
        'weight': [],
        'born_date': [],
        'born_city_country': [],
        'college': [],
        'high school': [],
        'team': [],
        'league': [],
        'nba_debut': [],
        'career_length': [],
        'points': [],
        'death_date': []
    })
    index = 0
    for player_id, player_info in to_scrape:
        try:
            try:
                clg = re.findall('College: ([\w\s,]*)', player_info)[0]
            except:
                try:
                    clg = re.findall('Colleges: ([\w\s,]*)', player_info)[0]
                except:
                    clg = None

            try:
                high_school = re.findall('High School: ([\w\s\.]*)in', player_info)[0]
            except:
                try:
                    high_school = re.findall('High Schools: ([\w\s\.]*)in', player_info)[0]
                except:
                    high_school = None
            
            try:
                career_length = re.findall('Career Length: ([\w\s]*year)', player_info)[0]
            except:
                try:
                    career_length = re.findall('Experience: ([\w\s]*year)', player_info)[0]
                except:
                    career_length = re.findall('Experience: ([\w\s]*Rookie)', player_info)[0]

            try:
                team = re.findall('Draft: ([\w\s\.]*)', player_info)[0]
            except:
                team = None
            
            death_date = None
            age = None
            try:
                age = re.findall('Age: (\d+)', player_info)[0]
            except:
                try:
                    death_date = re.findall('Died: (\w+ \d*,* \d*)', player_info)[0]
                except:
                    age = None

            try:
                nba_debut = re.findall('NBA Debut: (\w* \d*, \d*)', player_info)[0]
                league = 'NBA'
            except:
                try:
                    nba_debut = re.findall('ABA Debut: (\w* \d*, \d*)', player_info)[0]
                    league = 'ABA'
                except:
                    nba_debut = None
                    league = None

            player = {
            'player_id': [player_id],
            'position': [re.findall('Position: ([\w\s]*)', player_info)[0]],
            'shoots': [re.findall('Shoots: (\w+\s*\w*)\n', player_info)[0]],
            'height': [re.findall('\((\d+)cm, (\d+)kg\)', player_info)[0][0]],
            'weight': [re.findall('\((\d+)cm, (\d+)kg\)', player_info)[0][1]],
            'born_date': [re.findall('Born: (\w+ \d*,*\s*\d*)', player_info)[0]],
            'age': [age],
            'born_city_country': [re.findall(' in ([\w\s,\.-]*)\n', player_info)[0]],
            'college': [clg],
            'high school': [high_school],
            'team': [team],
            'league': [league],
            'nba_debut': [nba_debut],
            'career_length': [career_length],
            'points': [re.findall('PTS\n([\d\.]*)', player_info)[0]],
            'death_date': [death_date]
            }
            
            players_details = pd.concat([players_details, pd.DataFrame(player)], ignore_index=True)
            index += 1
        except:
            continue
    return players_details

creating chunks

In [ ]:
base_cut = np.array([0, 300, 600, 900, 1200, 1500, 1800])
cut1 = base_cut + 1800
cut2 = np.array([0, 300, 600, 900, 1200, 1808]) + 3600

cuts = [base_cut, cut1, cut2]

getting data and extracting fields

In [ ]:
for index, cut in enumerate(cuts):
    to_scrape, didnt_fetch = get_player_details(cut)
    extract_player_details(to_scrape).to_csv(f'player-details{index}.csv', index=False)

## NBA Michael Jordan Award Winners

this data will be fetched from this [page](https://www.basketball-reference.com/awards/mvp.html).

In [ ]:
driver = webdriver.Chrome()

driver.get('https://www.basketball-reference.com/awards/mvp.html')

share_button = driver.find_element(By.XPATH, '//*[@id="mvp_NBA_sh"]/div/ul/li[1]')
ActionChains(driver).move_to_element(share_button).perform()

csv_button = driver.find_elements(By.CLASS_NAME, 'tooltip')[2]
csv_button.click()

mvp = driver.find_element(By.ID, 'csv_mvp_NBA')
mvp = mvp.get_attribute('innerHTML')

mvp_csv = mvp[mvp.find('Season'):]

In [ ]:
file = open('mvp.csv', 'w+', encoding='utf:8')
file.write(mvp_csv.replace('-9999', 'player_id'))

7014

## Season Ranking

for each season data will be fetched from a page like [this](https://www.basketball-reference.com/leagues/NBA_2024_totals.html#totals_stats::pts).

In [ ]:
chrome_options = Options()
chrome_options.page_load_strategy = 'none'
chrome_options.add_argument("--headless")
chrome_options.add_argument('--user-agent="Mozilla/5.0 (Windows Phone 10.0; Android 4.2.1; Microsoft; Lumia 640 XL LTE) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/42.0.2311.135 Mobile Safari/537.36 Edge/12.10166"')
driver = webdriver.Chrome(options=chrome_options)

slected_cols = ['Rk', 'Player', 'Age', 'Team', 'Pos', 'G', 'FG', 'FGA', 'FG%', 'FT', 'FTA', 'FT%', 'AST', 'PF', 'PTS', 'Trp-Dbl', 'Awards', 'Player-additional']

seasons_top_player_csv = ''
years = range(1950, 2027)
index = 1950

while index < 2027:

    try:
        driver.get(f'https://www.basketball-reference.com/leagues/NBA_{index}_totals.html')
        driver.implicitly_wait(2)

        WebDriverWait(driver, 40).until(EC.presence_of_element_located((By.XPATH, '//*[@id="totals_stats_sh"]/div[2]/ul/li[1]')))
        element = driver.find_element(By.XPATH, '//*[@id="totals_stats"]/tbody/tr[7]/td[1]')
        share_button = driver.find_element(By.XPATH, '//*[@id="totals_stats_sh"]/div[2]/ul/li[1]')
        
        ActionChains(driver).move_to_element(element).perform()
        ActionChains(driver).move_to_element(share_button).perform()

        csv_button = driver.find_elements(By.CLASS_NAME, 'tooltip')[2]
        csv_button.click()

        stats = driver.find_element(By.ID, 'csv_totals_stats')
        stats = stats.get_attribute('innerHTML')

        file = open('temp.txt', 'w+', encoding='utf:8')
        file.write(stats[stats.find('Rk'):])
        file.close()

        temp = pd.read_csv('temp.txt')
        temp = temp[slected_cols]
        temp['Year'] = index
        temp = temp[:-1]
        temp.to_csv('temp.txt', index=False) if index == 1950 else temp.to_csv('temp.txt', header=False, index=False)

        file = open('temp.txt', 'r', encoding='utf:8')
        seasons_top_player_csv += file.read()

        index += 1

    except:
        continue

In [ ]:
file = open('seasons_top_player.csv', 'w+', encoding='utf:8')
file.write(seasons_top_player_csv)

3272446

## Adding ID for Tables

We will add id for players, colleges and high schools.

In [ ]:
players = pd.read_csv('players.csv')

mvp = pd.read_csv('mvp.csv')

players_details1 = pd.read_csv('player_details1.csv')
players_details2 = pd.read_csv('player-details2.csv')
players_details3 = pd.read_csv('player-details3.csv')

top_players = pd.read_csv('seasons_top_player.csv')

renaming for making joins simpler, dropping extra columns

In [ ]:
mvp.rename({'player_id': 'Player-additional'}, axis=1, inplace=True)
mvp.drop(columns='Tm', inplace=True)

In [ ]:
players_details1.rename({'player_id': 'Player-additional'}, axis=1, inplace=True)
players_details2.rename({'player_id': 'Player-additional'}, axis=1, inplace=True)
players_details3.rename({'player_id': 'Player-additional'}, axis=1, inplace=True)

adding player id

In [ ]:
players = players.reset_index().rename({'index': 'Player_Id'}, axis=1)

removing wrong rows

In [ ]:
players.drop(players[players['Player'] == 'Player'].index, inplace=True)

saving new data

In [ ]:
players.to_csv('player_ids.csv', index=False)

using player ids for other tabels

In [ ]:
id_map = players[['Player_Id', 'Player-additional']]

top_players.merge(id_map, on='Player-additional', how='left').to_csv('top_players_ids.csv', index=False)
mvp.merge(id_map, on='Player-additional', how='left').to_csv('mvp_ids.csv', index=False)

concating player details data

In [ ]:
all_player_details = pd.concat([players_details1, players_details2, players_details3]).drop_duplicates()

In [ ]:
all_player_details

,Player-additional,position,shoots,height,weight,born_date,born_city_country,college,high school,team,league,nba_debut,career_length,points,death_date,age
0,blankla01,Shooting Guard,Right,193,86,"September 9, 1966","Del Rio, Texas us",Virginia,McCullough,Detroit Pistons,NBA,"November 2, 1990",3 year,2.0,"May 3, 2023",NaN
1,champmi01,Small Forward,Right,208,104,"April 5, 1964","Everett, Washington us",Gonzaga,Everett,NaN,NBA,"March 3, 1989",1 year,0.0,NaN,61.0
2,crossru01,Center,Right,208,97,"September 5, 1961","Chicago, Illinois us",Purdue,Manley Career Academy,Golden State Warriors,NBA,"November 12, 1983",1 year,3.7,NaN,64.0
3,brownmo01,Center,Right,218,117,"October 13, 1999","New York, New York us",UCLA,Archbishop Molloy,NaN,NBA,"November 10, 2019",6 year,5.3,NaN,26.0
4,abdelal01,Power Forward,Right,208,108,"June 24, 1968","Cairo, Egypt eg",Duke,Bloomfield,Portland Trail Blazers,NBA,"November 2, 1990",5 year,5.7,NaN,57.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1775,sandela01,Center and Power Forward,Right,211,106,"November 21, 1988","Fort Pierce, Florida us",VCU,Port St. Lucie,Milwaukee Bucks,NBA,"October 27, 2010",6 year,6.4,NaN,NaN
1776,sandeme01,Shooting Guard,Right,196,95,"January 3, 1981","Liberal, Kansas us",Seward,Liberal,NaN,NBA,"November 9, 2005",1 year,2.6,NaN,45.0
1777,sandemi01,Small Forward,Right,198,95,"May 7, 1960","Vidalia, Louisiana us",UCLA,DeRidder,Kansas City Kings,NBA,"February 10, 1983",11 year,8.0,NaN,65.0
1778,sandeto01,Small Forward and Power Forward,Right,198,95,"November 8, 1938","New York, New York us",NYU,Seward Park,Boston Celtics,NBA,"October 22, 1960",13 year,9.6,NaN,87.0


using team id to add team id for player details table

In [ ]:
teams = pd.read_excel('tems.xlsx')[['team_id', 'franchise_name', 'names']]

extracting old names to make sure all players with team name will have team id

In [ ]:
teams['names'] = teams['names'].apply(lambda x: json.loads(x.replace("'", '"')))
teams = teams.explode('names', ignore_index=True)
teams.loc[teams['names'].isna(), 'names'] = teams.loc[teams['names'].isna(), 'franchise_name']

In [ ]:
teams

,team_id,franchise_name,names
0,1,Atlanta Hawks,Atlanta Hawks
1,1,Atlanta Hawks,St. Louis Hawks
2,1,Atlanta Hawks,Milwaukee Hawks
3,1,Atlanta Hawks,Tri-Cities Blackhawks
4,2,Boston Celtics,Boston Celtics
...,...,...,...
94,51,Virginia Squires,Virginia Squires
95,51,Virginia Squires,Washington Capitols
96,51,Virginia Squires,Oakland Oaks
97,52,Washington Capitols,Washington Capitols


In [ ]:
all_player_details = all_player_details.merge(teams, left_on='team', right_on='names', how='left').drop(columns='names')
all_player_details = all_player_details.merge(id_map, on='Player-additional', how='left')

making sure joins did not create duplicates

In [ ]:
all_player_details.drop_duplicates(subset=['Player-additional']).to_csv('player_details_ids.csv', index=False)

## Seperating Tables

reading data

In [ ]:
mvp = pd.read_csv('mvp_ids.csv')
player_detail = pd.read_csv('player_details_ids.csv')
player = pd.read_csv('player_ids.csv')
seasons = pd.read_csv('top_players_ids.csv')

In [ ]:
player_detail.columns

Index(['Player-additional', 'position', 'shoots', 'height', 'weight',
       'born_date', 'born_city_country', 'college', 'high school', 'team',
       'league', 'nba_debut', 'career_length', 'points', 'death_date', 'age',
       'team_id', 'franchise_name', 'Player_Id'],
      dtype='object')

joining player information

In [ ]:
player_all = player[['Player_Id', 'Player', 'From', 'To']].merge(player_detail, on=['Player_Id'], how='left')

seperating colleges

In [ ]:
player_college = player_all[['Player_Id', 'college']].explode(column='college')
colleges = pd.DataFrame(player_all['college'].unique()).reset_index().rename(columns={'index': 'college_id', 0: 'college_name'})
player_college = player_college.merge(colleges, left_on='college', right_on='college_name')[['Player_Id', 'college_id']]

seperating high schools

In [ ]:
player_highschool = player_all[['Player_Id', 'high school']].explode(column='high school')
high_schools = pd.DataFrame(player_all['high school'].unique()).reset_index().rename(columns={'index': 'high_school_id', 0: 'high_school_name'})
player_highschool = player_highschool.merge(high_schools, left_on='high school', right_on='high_school_name')[['Player_Id', 'high_school_id']]

In [3]:
player_all = pd.read_csv('data/player.csv')

seperating positions

In [39]:
position_split = player_all[['Player_Id', 'position']]
position_split.loc[:, 'position'] = position_split['position'].str.split('and')
position_split = position_split.explode('position')
position_split['position'] = position_split['position'].str.strip()

In [52]:
positions = pd.DataFrame(position_split['position'].unique()).dropna().reset_index().rename(columns={'index':'position_id', 0:'position_name'})

In [53]:
positions

,position_id,position_name
0,0,Power Forward
1,1,Center
2,2,Point Guard
3,3,Shooting Guard
4,4,Small Forward
5,5,Guard
6,6,Forward


In [57]:
player_position = position_split.merge(positions, left_on='position', right_on='position_name')[['Player_Id', 'position_id']]

In [58]:
player_position

,Player_Id,position_id
0,0,0
1,1,1
2,1,0
3,2,1
4,3,2
...,...,...
6752,5403,1
6753,5404,1
6754,5405,2
6755,5406,1


removing extra columns from players

In [ ]:
player_all.drop(columns=['franchise_name', 'team', 'high school', 'college', 'position'], inplace=True)

saving data

In [ ]:
player_all.to_csv('player.csv', index=False)
colleges.to_csv('colleges.csv', index=False)
high_schools.to_csv('high_schools.csv', index=False)
player_college.to_csv('player_college.csv', index=False)
player_highschool.to_csv('player_high_school.csv', index=False)
player_position.to_csv('player_position.csv', index=False)
positions.to_csv('positions.csv', index=False)